<a href="https://colab.research.google.com/github/Thilac01/Statistical-Learning-e22395/blob/main/Statistical_Learning_Assignement5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Gaussian Process Regression

#Fetch Data From Kaggle

In [ ]:
import kagglehub

# Download latest version
kagglepath="elikplim/eergy-efficiency-dataset"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

#Load the data in this Env

In [ ]:
import pandas as pd
import os
print(f"Listing contents of: {path}")
!ls {path}
df2=pd.read_csv(path+"/ENB2012_data.csv")

In [ ]:
df2

#Data Preprocessing & Inspection

In [ ]:
pip install ezclean


#Clean The data using My Lib

In [ ]:
from ezclean import*
import numpy as np

df2 = Cleaner(df2)

df2

In [ ]:

data_matrix = df2[['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8']]
y_heating = df2['y1']
y_cooling = df2['y2']

In [ ]:
import numpy as np

# 3. Split into training (80%) and testing (20%) datasets manually
np.random.seed(42)
n_samples = len(data_matrix)
shuffled_indices = np.random.permutation(n_samples)
split_idx = int(n_samples * 0.8)

train_indices = shuffled_indices[:split_idx]
test_indices = shuffled_indices[split_idx:]

x_train = data_matrix.iloc[train_indices].values
y1_train = y_heating.iloc[train_indices].values
y2_train = y_cooling.iloc[train_indices].values

x_test = data_matrix.iloc[test_indices].values
y1_test = y_heating.iloc[test_indices].values
y2_test = y_cooling.iloc[test_indices].values

# 4. Standardize the features using only NumPy
x_mean = np.mean(x_train, axis=0)
x_std = np.std(x_train, axis=0)

x_train_scaled = (x_train - x_mean) / x_std
x_test_scaled = (x_test - x_mean) / x_std

print("Data Cleaning and Splitting complete.")
print(f"Train shapes - Features: {x_train_scaled.shape}, y1: {y1_train.shape}, y2: {y2_train.shape}")

#Make Kernel Function

In [ ]:
# 5. Define the RBF Kernel function
def rbf_kernel(x_a, x_b, length_scale=1.5, sigma_f=1.0):
    """
    Computes the Radial Basis Function (RBF) covariance matrix.

    Parameters:
    x_a: Matrix of shape (N, D) representing features x1 to x8
    x_b: Matrix of shape (M, D) representing features x1 to x8
    """
    # Matrix-compatible pairwise squared Euclidean distance calculation
    sq_dist = np.sum(x_a**2, axis=1).reshape(-1, 1) + np.sum(x_b**2, axis=1) - 2 * np.dot(x_a, x_b.T)
    return (sigma_f ** 2) * np.exp(-0.5 / (length_scale ** 2) * sq_dist)



#Gaussian Process Regression and Prediction

In [ ]:
# GP Prediction function utilizing our defined kernel
def gpr_predict(x_train, y_train, x_test, length_scale=1.5, sigma_f=1.0, sigma_n=1e-1):
    n_train = len(x_train)

    # Calculate covariance matrices using our kernel
    K = rbf_kernel(x_train, x_train, length_scale, sigma_f) + (sigma_n ** 2) * np.eye(n_train)
    K_s = rbf_kernel(x_train, x_test, length_scale, sigma_f)
    K_ss = rbf_kernel(x_test, x_test, length_scale, sigma_f)

    # Stable matrix solution using Cholesky Decomposition
    L = np.linalg.cholesky(K)

    # Calculate predictive mean (mu)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y_train))
    mu = np.dot(K_s.T, alpha)

    # Calculate predictive standard deviation (uncertainty)
    v = np.linalg.solve(L, K_s)
    variance = np.diag(K_ss) - np.sum(v**2, axis=0)

    return mu, np.sqrt(variance)

# Manual Evaluation Metric Functions
def compute_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

def compute_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

# Generate predictions for y1 and y2
pred_y1, std_y1 = gpr_predict(x_train_scaled, y1_train, x_test_scaled)
pred_y2, std_y2 = gpr_predict(x_train_scaled, y2_train, x_test_scaled)

# Output final evaluations
print("--- Heating Load (y1) Performance ---")
print(f"R² Score: {compute_r2(y1_test, pred_y1):.4f}")
print(f"RMSE: {compute_rmse(y1_test, pred_y1):.4f}")

print("\n--- Cooling Load (y2) Performance ---")
print(f"R² Score: {compute_r2(y2_test, pred_y2):.4f}")
print(f"RMSE: {compute_rmse(y2_test, pred_y2):.4f}")

#Plotting Performance & Error Analysis

In [ ]:
import matplotlib.pyplot as plt

# Generate performance visualization graphs
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: True vs Predicted for Heating Load (y1) with standard error bars
axes[0].errorbar(y1_test, pred_y1, yerr=std_y1, fmt='o', color='crimson', ecolor='gray', alpha=0.7, label='Predictions with $\sigma$')
axes[0].plot([y1_test.min(), y1_test.max()], [y1_test.min(), y1_test.max()], 'k--', lw=2, label='Perfect Prediction')
axes[0].set_title(f'Heating Load ($y_1$) Model Evaluation\n$R^2$: {compute_r2(y1_test, pred_y1):.3f}')
axes[0].set_xlabel('True Target Values')
axes[0].set_ylabel('Predicted Target Values')
axes[0].legend()
axes[0].grid(True, linestyle=':')

# Plot 2: True vs Predicted for Cooling Load (y2) with standard error bars
axes[1].errorbar(y2_test, pred_y2, yerr=std_y2, fmt='o', color='royalblue', ecolor='gray', alpha=0.7, label='Predictions with $\sigma$')
axes[1].plot([y2_test.min(), y2_test.max()], [y2_test.min(), y2_test.max()], 'k--', lw=2, label='Perfect Prediction')
axes[1].set_title(f'Cooling Load ($y_2$) Model Evaluation\n$R^2$: {compute_r2(y2_test, pred_y2):.3f}')
axes[1].set_xlabel('True Target Values')
axes[1].set_ylabel('Predicted Target Values')
axes[1].legend()
axes[1].grid(True, linestyle=':')

plt.tight_layout()
plt.show()

## Discussion & Conclusions

Based on our structured step-by-step custom implementation utilizing basic NumPy calculations:

1. **Model Stability:** Running the clean up phase first guarantees that invalid data points do not break our custom linear solvers during the matrix decomposition step.
2. **Predictive Capability:** Applying the feature parameters ($x_1$ through $x_8$) straight through the custom-made RBF kernel yields strong predictive efficiency ($R^2 > 0.95$). This implies that the patterns governing thermodynamic responses are smooth and well-captured by single-parameter processes.
3. **Output Correlations:** While evaluating heating load ($y_1$) and cooling load ($y_2$) as independent processes yields clear predictions and standard deviation bounds, setting up a shared multi-task covariance structure in the future could enhance the model by mapping their structural cross-correlation.

----
----

#Linear Regression

In [ ]:
import kagglehub

# Download latest version
kagglepath="programmer3/green-building-multi-source-environment-dataset" #"ujjwalchowdhury/energy-efficiency-data-set"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

In [ ]:

import os
print(f"Listing contents of: {path}")
!ls {path}
df2=pd.read_csv(path+"/green_building_dataset.csv")
df2 = Cleaner(df2)
df2

In [ ]:
feature_cols = [
    'electricity_consumption',
    'heating_energy',
    'cooling_energy',
    'indoor_temperature',
    'outdoor_temperature',
    'equipment_load'
]
target_col = 'predicted_energy_demand'

# Convert features and targets to clean NumPy arrays
x_data = df2[feature_cols].to_numpy()
y_data = df2[target_col].to_numpy()

# 3. Handle Train-Test Split (80% Train, 20% Test) manually using NumPy indices
np.random.seed(42)
n_samples = len(x_data)
shuffled_idx = np.random.permutation(n_samples)
split_idx = int(n_samples * 0.8)

train_idx = shuffled_idx[:split_idx]
test_idx = shuffled_idx[split_idx:]

x_train = x_data[train_idx]
y_train = y_data[train_idx]

x_test = x_data[test_idx]
y_test = y_data[test_idx]

# 4. Standardize features to protect matrix conditions
x_mean = np.mean(x_train, axis=0)
x_std = np.std(x_train, axis=0)

x_train_scaled = (x_train - x_mean) / x_std
x_test_scaled = (x_test - x_mean) / x_std

print("Data cleaning and parameter partitioning complete.")
print(f"Train shapes: Features {x_train_scaled.shape}, Target {y_train.shape}")
print(f"Test shapes: Features {x_test_scaled.shape}, Target {y_test.shape}")

In [ ]:
# Linear Regression via Normal Equations: w = (X^T * X)^-1 * X^T * y
def fit_linear_regression(x, y):
    # Add a column of ones to represent the intercept term (bias)
    n_samples = x.shape[0]
    X_bias = np.hstack((np.ones((n_samples, 1)), x))

    # Solve normal equations analytically
    w = np.linalg.inv(X_bias.T @ X_bias) @ X_bias.T @ y
    return w

def predict_linear_regression(x, w):
    n_samples = x.shape[0]
    X_bias = np.hstack((np.ones((n_samples, 1)), x))
    return X_bias @ w

# Model training
weights = fit_linear_regression(x_train_scaled, y_train)

# Model inference
pred_y = predict_linear_regression(x_test_scaled, weights)

# Performance calculation metrics from scratch
def compute_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

def compute_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

r2_val = compute_r2(y_test, pred_y)
rmse_val = compute_rmse(y_test, pred_y)

print("--- Linear Regression Performance Metrics ---")
print(f"Intercept (Bias): {weights[0]:.4f}")
print("Feature weights (w1 to w6):", np.round(weights[1:], 4))
print(f"R² Score: {r2_val:.4f}")
print(f"RMSE: {rmse_val:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Generate evaluation visuals
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Plot 1: Prediction vs True Value Scatter
axes[0].scatter(y_test, pred_y, color='teal', alpha=0.6, edgecolors='k')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Fit Line')
axes[0].set_title(f'Actual vs. Predicted Energy Demand\n$R^2$: {r2_val:.3f}')
axes[0].set_xlabel('True Energy Demand ($y$)')
axes[0].set_ylabel('Predicted Energy Demand ($\hat{y}$)')
axes[0].legend()
axes[0].grid(True, linestyle=':')

# Plot 2: Residual Analysis Distribution
residuals = y_test - pred_y
axes[1].hist(residuals, bins=25, color='darkorange', edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--', lw=2)
axes[1].set_title('Residuals Error Distribution')
axes[1].set_xlabel('Residual Error Value ($y - \hat{y}$)')
axes[1].set_ylabel('Frequency Count')
axes[1].grid(True, linestyle=':')

plt.tight_layout()
plt.show()

## Discussion & Conclusions

Based on our direct implementation using analytical matrix updates, we draw the following conclusions:

1. **Parameter Justification:** Selecting raw subsystem measurements (heating, cooling, and electrical components) alongside baseline environmental indices provides an explicit link to structural power demands. The standardized feature values ($w_1$ to $w_6$) reveal which mechanical attributes exert the greatest continuous leverage over total target consumption ($y$).
2. **Analysis of Performance plots:** The Actual vs. Predicted graph shows that the points align very closely with the 45-degree target line, which confirms a high $R^2$ performance score. Additionally, the residual distribution plot is centered closely around zero, showing that the model's errors are stable and symmetrical.
3. **Linear Approximations:** Because total energy demand is a cumulative sum of independent architectural loads (heating + cooling + basic utilities), the baseline system structure is highly linear. Therefore, standard closed-form Ordinary Least Squares estimation works excellently without needing complex regularizations or deep layers.